In [16]:
library(dplyr)
library(purrr)
library(WGCNA)

In [17]:
cohort <- "WHITE"
if (cohort == "TAIWAN") {
    config <- list(
        wd = "/home/seba/github_repos/crc_weighted_network/taiwanese_cohort/count_matrices_by_geneid",
        rna_base_counts = "RNA_merged_counts_filtered.csv",
        mirna_base_counts = "miRNA_merged_counts_filtered.csv",
        row_names_in_metadata = 1,
        metadata_file = "all_metadata_taiwan.csv",
        col_for_datatype = "Assay.Type",
        rna_datatype = "RNA-Seq",
        mirna_datatype = "miRNA-Seq",
        external_col = "PHENOTYPE",
        contrast_for_dea = c("PHENOTYPE", "neoplastic", "adjacent normal"),
        rna_network_dir = "rna_pooled_tmm",
        mirna_network_dir = "mirna_pooled_tmm",
        rna_kme_limit = 0,
        rna_kme_p_limit = 1,
        rna_gs_limit = 0.4,
        rna_gs_p_limit = 1e-02,
        mirna_kme_limit = 0,
        mirna_kme_p_limit = 1,
        mirna_gs_limit = 0.4,
        mirna_gs_p_limit = 1e-02
    )
} else if (cohort == "WHITE") {
    config <- list(
        wd = "/home/seba/github_repos/crc_weighted_network/cohort_white/count_matrices_white_byFILENAME",
        rna_base_counts = "RNA_merged_filtered.csv",
        mirna_base_counts = "mirna_new_filtered_merged.csv",
        row_names_in_metadata = 2,
        metadata_file = "gdc_sample_sheet.2026-01-21_GENES+MIRNAISOFORMS.csv",
        col_for_datatype = "Data.Type",
        rna_datatype = "Gene Expression Quantification",
        mirna_datatype = "Isoform Expression Quantification",
        external_col = "Tissue.Type",
        contrast_for_dea = c("Tissue.Type", "Tumor", "Normal"),
        rna_network_dir = "sampled_rna_pooled_tmm",
        mirna_network_dir = "newmirna_pooled",
        rna_kme_limit = 0,
        rna_kme_p_limit = 1,
        rna_gs_limit = 0.4,
        rna_gs_p_limit = 1e-02,
        mirna_kme_limit = 0,
        mirna_kme_p_limit = 1,
        mirna_gs_limit = 0.4,
        mirna_gs_p_limit = 1e-02
    )
}

In [18]:
# Función ajustada para buscar el máximo positivo
extract_top_performance <- function(iter_path) {
  # Cargar archivos
  MEs_path <- file.path(iter_path, "MEs.rds")
  traits_path <- file.path(iter_path, "datTraits_clean.rds")
  
  if (!file.exists(MEs_path) | !file.exists(traits_path)) return(NULL)

  MEs    <- readRDS(MEs_path)
  traits <- readRDS(traits_path)

  # Match samples
  common <- intersect(rownames(MEs), rownames(traits))
  MEs    <- MEs[common, , drop = FALSE]
  traits <- traits[common, , drop = FALSE]

  # Validación de fenotipo (Cáncer vs Normal)
  if (!(config$external_col %in% colnames(traits))) stop("Columna externa no encontrada.")
  trait_pheno <- traits[, config$external_col, drop = FALSE]

  # Conversión a numérico (Normal=0, Tumor=1)
  if (!is.numeric(trait_pheno[[config$external_col]])) {
    lev <- sort(unique(as.character(trait_pheno[[config$external_col]])))
    # Forzamos que el nivel 2 (alfabéticamente posterior, ej. 'Tumor') sea 1
    trait_pheno[[config$external_col]] <- ifelse(as.character(trait_pheno[[config$external_col]]) == lev[2], 1, 0)
  }

  # Correlación de Pearson
  MEtraitCor <- cor(MEs, trait_pheno, use = "p")
  
  # --- LÓGICA DE SELECCIÓN POSITIVA ---
  
  # 1. Buscamos el valor máximo (sin abs()). 
  # Esto nos dará el número positivo más alto.
  max_cor <- max(MEtraitCor, na.rm = TRUE)
  
  # 2. Identificamos qué módulo tiene ese valor exacto
  idx_max <- which(MEtraitCor == max_cor, arr.ind = TRUE)
  
  # 3. Extraer el nombre del módulo
  best_mod_name <- rownames(MEtraitCor)[idx_max[1,1]]
  best_mod_clean <- gsub("ME", "", best_mod_name)

  return(data.frame(
    folder = basename(iter_path), 
    max_correlation = max_cor,
    best_module = best_mod_clean
  ))
}

In [19]:
# 1. Función para extraer los genes directamente del objeto de red
get_module_genes_from_rds <- function(iter_path, best_mod_name) {
  # Construimos la ruta al archivo
  net_file <- file.path(iter_path, "blockwise_net.rds")
  
  if (!file.exists(net_file)) {
    warning(paste("Archivo no encontrado en:", iter_path))
    return(NULL)
  }
  
  # Cargar la red
  net <- readRDS(net_file)
  
  # Extraer los nombres de los genes que pertenecen al módulo ganador
  # net$colors es un vector donde los nombres son los genes y los valores el módulo
  genes_del_modulo <- names(net$colors)[net$colors == best_mod_name]
  
  return(genes_del_modulo)
}

In [20]:
# This script is about selecting the best network from the sampling
setwd("/home/seba/github_repos/crc_weighted_network/cohort_white/nets_from_sampling")


In [21]:
# List directories in the wd
dirs <- list.dirs()
# split directories in RNA and miRNA, exclude the ones that says cytoscape and modules
rna_dirs_raw <- dirs[grepl("_RNA_TMM_T", dirs)]
mirna_dirs_raw <- dirs[grepl("_newMIRNA_TMM_T", dirs)]

subdirs_pattern <- "/cytoscape|/modules"

rna_dirs <- rna_dirs_raw[!grepl(subdirs_pattern, rna_dirs_raw)]
mirna_dirs <- mirna_dirs_raw[!grepl(subdirs_pattern, mirna_dirs_raw)]

In [22]:
ranking_rna <- rna_dirs %>% 
  map_dfr(extract_top_performance) %>% 
  arrange(desc(max_correlation))

In [23]:
ranking_mirna <- mirna_dirs %>% 
  map_dfr(extract_top_performance) %>% 
  arrange(desc(max_correlation))

In [24]:
ranking_mirna

folder,max_correlation,best_module
<chr>,<dbl>,<chr>
10_newMIRNA_TMM_T_merged_counts,0.9033640,2
4_newMIRNA_TMM_T_merged_counts,0.8827363,2
7_newMIRNA_TMM_T_merged_counts,0.8532728,2
1_newMIRNA_TMM_T_merged_counts,0.8513516,2
5_newMIRNA_TMM_T_merged_counts,0.8500494,2
8_newMIRNA_TMM_T_merged_counts,0.8482134,2
2_newMIRNA_TMM_T_merged_counts,0.8446642,3
3_newMIRNA_TMM_T_merged_counts,0.7927568,2
6_newMIRNA_TMM_T_merged_counts,0.7674141,2


In [25]:
ranking_rna

folder,max_correlation,best_module
<chr>,<dbl>,<chr>
6_RNA_TMM_T_merged_counts,0.9071198,1
2_RNA_TMM_T_merged_counts,0.9041310,1
5_RNA_TMM_T_merged_counts,0.8841536,11
10_RNA_TMM_T_merged_counts,0.8839399,1
7_RNA_TMM_T_merged_counts,0.8737066,1
9_RNA_TMM_T_merged_counts,0.8706647,1
4_RNA_TMM_T_merged_counts,0.8508702,1
1_RNA_TMM_T_merged_counts,0.8393248,1
8_RNA_TMM_T_merged_counts,0.8347389,1


In [26]:
# 2. Recolectar todos los listados de genes (Usando tu ranking previo)
message("Cargando genes de las 10 redes... esto puede tardar un poco.")
gene_sets_list <- list()

for(i in 1:nrow(ranking_rna)) {
  folder_name <- ranking_rna$folder[i]
  path <- file.path(ranking_rna$folder[i])
  mod  <- ranking_rna$best_module[i]

  message(paste(">>> Procesando iteración", i, ":", folder_name))
  
  genes <- get_module_genes_from_rds(path, mod)
  gene_sets_list[[folder_name]] <- genes
}

# 3. Calcular Matriz de Similitud de Jaccard
n_sets <- length(gene_sets_list)
similarity_matrix <- matrix(0, nrow = n_sets, ncol = n_sets)
rownames(similarity_matrix) <- names(gene_sets_list)
colnames(similarity_matrix) <- names(gene_sets_list)

for(i in 1:n_sets) {
  for(j in 1:n_sets) {
    set1 <- gene_sets_list[[i]]
    set2 <- gene_sets_list[[j]]
    
    # Índice de Jaccard: Intersección / Unión
    intersection <- length(intersect(set1, set2))
    union <- length(union(set1, set2))
    similarity_matrix[i,j] <- intersection / union
  }
}

# 4. Determinar la Red Ganadora por Consenso
# Calculamos el promedio de similitud de cada red contra todas las demás
consenso_scores <- rowMeans(similarity_matrix)
ranking_consenso <- data.frame(
  folder = names(consenso_scores),
  similarity_score = consenso_scores
) %>% arrange(desc(similarity_score))

# 5. Resultado Final
best_consensus_folder <- ranking_consenso$folder[1]
message("\n--- RESULTADO DE CONSENSO ---")
message("La red que más se parece al promedio del grupo es: ", best_consensus_folder)
message("Puntaje de similitud media: ", round(ranking_consenso$similarity_score[1], 4))

Cargando genes de las 10 redes... esto puede tardar un poco.

>>> Procesando iteración 1 : 6_RNA_TMM_T_merged_counts

>>> Procesando iteración 2 : 2_RNA_TMM_T_merged_counts

>>> Procesando iteración 3 : 5_RNA_TMM_T_merged_counts

>>> Procesando iteración 4 : 10_RNA_TMM_T_merged_counts

>>> Procesando iteración 5 : 7_RNA_TMM_T_merged_counts

>>> Procesando iteración 6 : 9_RNA_TMM_T_merged_counts

>>> Procesando iteración 7 : 4_RNA_TMM_T_merged_counts

>>> Procesando iteración 8 : 1_RNA_TMM_T_merged_counts

>>> Procesando iteración 9 : 8_RNA_TMM_T_merged_counts

>>> Procesando iteración 10 : 3_RNA_TMM_T_merged_counts


--- RESULTADO DE CONSENSO ---

La red que más se parece al promedio del grupo es: 2_RNA_TMM_T_merged_counts

Puntaje de similitud media: 0.5761



In [27]:
ranking_consenso

,folder,similarity_score
,<chr>,<dbl>
2_RNA_TMM_T_merged_counts,2_RNA_TMM_T_merged_counts,0.5760756
7_RNA_TMM_T_merged_counts,7_RNA_TMM_T_merged_counts,0.5693735
9_RNA_TMM_T_merged_counts,9_RNA_TMM_T_merged_counts,0.5683639
3_RNA_TMM_T_merged_counts,3_RNA_TMM_T_merged_counts,0.5584865
4_RNA_TMM_T_merged_counts,4_RNA_TMM_T_merged_counts,0.5524787
8_RNA_TMM_T_merged_counts,8_RNA_TMM_T_merged_counts,0.5522626
6_RNA_TMM_T_merged_counts,6_RNA_TMM_T_merged_counts,0.5317597
1_RNA_TMM_T_merged_counts,1_RNA_TMM_T_merged_counts,0.5313641
10_RNA_TMM_T_merged_counts,10_RNA_TMM_T_merged_counts,0.4330892


In [28]:
# 2. Recolectar todos los listados de genes (Usando tu ranking previo)
message("Cargando genes de las 10 redes... esto puede tardar un poco.")
gene_sets_list <- list()

for(i in 1:nrow(ranking_mirna)) {
  folder_name <- ranking_mirna$folder[i]
  path <- file.path(ranking_mirna$folder[i])
  mod  <- ranking_mirna$best_module[i]

  message(paste(">>> Procesando iteración", i, ":", folder_name))
  
  genes <- get_module_genes_from_rds(path, mod)
  gene_sets_list[[folder_name]] <- genes
}

# 3. Calcular Matriz de Similitud de Jaccard
n_sets <- length(gene_sets_list)
similarity_matrix <- matrix(0, nrow = n_sets, ncol = n_sets)
rownames(similarity_matrix) <- names(gene_sets_list)
colnames(similarity_matrix) <- names(gene_sets_list)

for(i in 1:n_sets) {
  for(j in 1:n_sets) {
    set1 <- gene_sets_list[[i]]
    set2 <- gene_sets_list[[j]]
    
    # Índice de Jaccard: Intersección / Unión
    intersection <- length(intersect(set1, set2))
    union <- length(union(set1, set2))
    similarity_matrix[i,j] <- intersection / union
  }
}

# 4. Determinar la Red Ganadora por Consenso
# Calculamos el promedio de similitud de cada red contra todas las demás
consenso_scores <- rowMeans(similarity_matrix)
ranking_consenso <- data.frame(
  folder = names(consenso_scores),
  similarity_score = consenso_scores
) %>% arrange(desc(similarity_score))

# 5. Resultado Final
best_consensus_folder <- ranking_consenso$folder[1]
message("\n--- RESULTADO DE CONSENSO ---")
message("La red que más se parece al promedio del grupo es: ", best_consensus_folder)
message("Puntaje de similitud media: ", round(ranking_consenso$similarity_score[1], 4))

Cargando genes de las 10 redes... esto puede tardar un poco.

>>> Procesando iteración 1 : 10_newMIRNA_TMM_T_merged_counts

>>> Procesando iteración 2 : 4_newMIRNA_TMM_T_merged_counts

>>> Procesando iteración 3 : 7_newMIRNA_TMM_T_merged_counts

>>> Procesando iteración 4 : 1_newMIRNA_TMM_T_merged_counts

>>> Procesando iteración 5 : 5_newMIRNA_TMM_T_merged_counts

>>> Procesando iteración 6 : 8_newMIRNA_TMM_T_merged_counts

>>> Procesando iteración 7 : 2_newMIRNA_TMM_T_merged_counts

>>> Procesando iteración 8 : 3_newMIRNA_TMM_T_merged_counts

>>> Procesando iteración 9 : 6_newMIRNA_TMM_T_merged_counts

>>> Procesando iteración 10 : 9_newMIRNA_TMM_T_merged_counts


--- RESULTADO DE CONSENSO ---

La red que más se parece al promedio del grupo es: 1_newMIRNA_TMM_T_merged_counts

Puntaje de similitud media: 0.6827



In [29]:
ranking_consenso

,folder,similarity_score
,<chr>,<dbl>
1_newMIRNA_TMM_T_merged_counts,1_newMIRNA_TMM_T_merged_counts,0.6826776
10_newMIRNA_TMM_T_merged_counts,10_newMIRNA_TMM_T_merged_counts,0.6783441
7_newMIRNA_TMM_T_merged_counts,7_newMIRNA_TMM_T_merged_counts,0.6714159
8_newMIRNA_TMM_T_merged_counts,8_newMIRNA_TMM_T_merged_counts,0.6697130
4_newMIRNA_TMM_T_merged_counts,4_newMIRNA_TMM_T_merged_counts,0.6654435
3_newMIRNA_TMM_T_merged_counts,3_newMIRNA_TMM_T_merged_counts,0.6631565
6_newMIRNA_TMM_T_merged_counts,6_newMIRNA_TMM_T_merged_counts,0.5202055
5_newMIRNA_TMM_T_merged_counts,5_newMIRNA_TMM_T_merged_counts,0.4818644
9_newMIRNA_TMM_T_merged_counts,9_newMIRNA_TMM_T_merged_counts,0.4501465
